# Hướng 2, bước 2: trích đầu ra **pooler** của probe Meta cho train và val (GPU)

Probe EK100 của Meta = attentive pooler (4 block, 3 query) → 3 lớp Linear (verb / noun / action). Notebook này chạy encoder +
predictor + **pooler đóng băng** trên từng clip và lưu đầu ra pooler (`pooled`, 3 × 1024 fp16, khoảng 6 KB mỗi clip) cùng logits của
probe Meta. Bước 3 (`kaggle_train_head.ipynb`) train lại 3 lớp Linear trên đó, không cần GPU nhiều.

- **Val** (9.3k clip): dùng bản 292p `ek100-292-val-0/1`. Logits ở đây chính là baseline của probe Meta.
- **Train**: dùng các bản 292p của tập train (bước 1). Chỉ những video có trong Input được liệt kê, nên chạy được khi mới có phần mở
  rộng; khi có thêm video EK55, chạy lại notebook với Input mới: các clip đã trích được bỏ qua (clip id cố định).
- `ANCHORS`: `action_start` (giao thức chuẩn EK100, dùng để so sánh chính) và `official` (giao thức của code Meta, để so với 32.7).
- `VIEWS`: 1 = mỗi segment 1 clip với thiết lập như val (anticipation 1 s). Thêm view (2, 3) = thêm clip với thời gian anticipation /
  điểm neo ngẫu nhiên theo đúng khoảng của config train của Meta (augmentation); mỗi view tốn thêm thời gian như view đầu.

**Thời gian đo được (2×T4, fp16; nút thắt là giải mã video trên CPU):** val khoảng 1.7–1.8 giờ mỗi anchor (0.7–0.8 clip/s mỗi GPU);
train phần mở rộng (37k clip) khoảng 4.7–5 giờ mỗi anchor mỗi view (khoảng 1.1 clip/s mỗi GPU). Toàn bộ (val + train, 2 anchor) cần 2 phiên:
notebook tự dừng ở `MAX_HOURS` để output được lưu. Output khoảng 0.5 GB mỗi 37k clip. Phiên bị ngắt: *Add Input* output của phiên đó rồi chạy lại, các clip đã xong được bỏ qua.

**Cài đặt:** GPU **T4 ×2**, Internet **On**. *Add Input*: `ek100-292-val-0`, `ek100-292-val-1`, và mọi dataset `ek100-292-train-*`.

In [ ]:
import time
NOTEBOOK_START = time.time()
ANCHORS = ["action_start", "official"]
VIEWS = 1
FIX_FRAME_IDS = False   # False = như code Meta (khớp các con số 32.7 / 12.85 trên bản 292p)
MAX_HOURS = 11.0        # dừng sạch trước giới hạn 12 giờ của Kaggle (output được lưu); chạy lại để làm tiếp

In [ ]:
BRANCH = "direction2-balanced-head"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /tmp/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /tmp/vjepa2 && cd /tmp/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops av
import os
os.environ["VJEPA2_ROOT"] = "/tmp/vjepa2"
R = "/tmp/repo"
!git -C $R log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
# annotations (the full train csv: probe classes are derived from it)
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /tmp/ek100-ann
ANN = "/tmp/ek100-ann"

In [ ]:
!mkdir -p /tmp/ckpt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/vitl.pt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/evals/ek100-vitl-256.pt
!ls -lh /tmp/ckpt

In [ ]:
# Resume + clip indices (val: prepare_ek100.py, train: prepare_train_clips.py)
import glob, os, shutil, json
OUT = "/kaggle/working/feats_pooled"
prev = glob.glob("/kaggle/input/**/feats_pooled", recursive=True)
for d in prev:
    shutil.copytree(d, OUT, dirs_exist_ok=True); print("resumed from", d)
mp4 = glob.glob("/kaggle/input/**/ek100_292/**/*.MP4", recursive=True)
print(f"{len(mp4)} re-encoded videos attached")
W = "/kaggle/working/work"
!python $R/scripts/prepare_ek100.py --video_root /kaggle/input --train_csv $ANN/EPIC_100_train.csv \
    --val_csv $ANN/EPIC_100_validation.csv --video_info $ANN/EPIC_100_video_info.csv --out_dir $W | grep -E '"val_clips"|val_videos_found'
!python $R/scripts/prepare_train_clips.py --video_root /kaggle/input --train_csv $ANN/EPIC_100_train.csv \
    --val_csv $ANN/EPIC_100_validation.csv --video_info $ANN/EPIC_100_video_info.csv --out_dir $W --views $VIEWS
import pandas as pd
for csv in ("val_clips.csv", "train_clips.csv"):
    c = pd.read_csv(f"{W}/{csv}")
    other = c[~c.video_path.str.contains("/ek100_292/")].video_id.unique()
    assert not len(other), f"{csv}: {len(other)} videos are not 292p copies (e.g. {other[:3]}): remove other video datasets from Input"
    print(csv, len(c), "clips,", c.video_id.nunique(), "videos, all 292p copies")

In [ ]:
# One process per GPU, val first (small), then train. Every 256 clips are saved; a rerun skips what is done.
# The whole loop stops at MAX_HOURS (workers killed, at most ~255 clips per GPU lost) so the session ends normally
# and its output is kept: attach that output and run again to continue.
import os, signal, subprocess, time
FIX = "--fix_frame_ids" if FIX_FRAME_IDS else ""
def hours():
    return (time.time() - NOTEBOOK_START) / 3600
def shard_cmd(gpu, split, anchor):
    csv = "val_clips.csv" if split == "val" else "train_clips.csv"
    run = (f"python {R}/scripts/eval_ek100.py --work_dir {W} --clips_csv {csv} --out_dir {OUT}/{split} "
           f"--vitl_ckpt /tmp/ckpt/vitl.pt --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt --anchors {anchor} "
           f"--device cuda:{gpu} --shard_id {gpu} --num_shards 2 --dtype fp16 --batch_size 4 --num_workers 2 "
           f"--save_pooled --decode_fallback pyav {FIX}")
    return f"for i in 1 2 3 4 5; do {run} && break; echo \"[retry $i]\"; done"

STOPPED = False
for split in ("val", "train"):
    for anchor in ANCHORS:
        if STOPPED or hours() > MAX_HOURS - 0.3:
            STOPPED = True
            print(f"== skipping {split} {anchor}: time budget", flush=True)
            continue
        t0 = time.time()
        procs = {g: subprocess.Popen(["bash", "-c", shard_cmd(g, split, anchor) + f" >> /kaggle/working/{split}_{anchor}_gpu{g}.log 2>&1"],
                                     start_new_session=True) for g in (0, 1)}
        while any(p.poll() is None for p in procs.values()):
            for _ in range(30):
                if all(p.poll() is not None for p in procs.values()) or hours() > MAX_HOURS:
                    break
                time.sleep(10)
            if hours() > MAX_HOURS:
                for p in procs.values():
                    if p.poll() is None:
                        os.killpg(p.pid, signal.SIGTERM)
                time.sleep(20)
                STOPPED = True
                print(f"== {split} {anchor}: stopped at {hours():.2f} h (time budget). Attach this output and run again.", flush=True)
                break
            for g in procs:
                lines = [l for l in open(f"/kaggle/working/{split}_{anchor}_gpu{g}.log").read().splitlines()
                         if any(k in l for k in ("clip/s", "to go", "[error]", "[retry", "Traceback"))]
                print(f"{split} {anchor} gpu{g}: {lines[-1] if lines else 'running...'}", flush=True)
        print(f"== {split} {anchor}: exit codes {[p.returncode for p in procs.values()]}, {(time.time() - t0) / 3600:.2f} h", flush=True)
!du -sh $OUT/*

In [ ]:
# Baseline of the released probe on the val copies (should be close to 32.20 official / 12.85 official on 292p)
for anchor in ANCHORS:
    if not glob.glob(f"{OUT}/val/{anchor}/shard*/chunk_*.npz"):
        continue
    !python $R/scripts/compute_metrics.py --work_dir $W --logits_dir $OUT/val/{anchor} --out /kaggle/working/meta_val_{anchor}.json --allow_partial
import numpy as np
n = sum(len(np.load(f)["clip_id"]) for f in glob.glob(f"{OUT}/train/*/shard*/chunk_*.npz"))
print("train clips extracted (all anchors):", n)

Xong thì tạo **Dataset từ Output** (thư mục `feats_pooled` và `work`), ví dụ `ek100-feats-pooled`, rồi dùng ở bước 3.